## 📑 Table of Contents

* [Why This Notebook Is Non-Negotiable](#why-this-notebook-is-non-negotiable)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Two Patterns: Sync vs Async Job Handling](#1-the-two-patterns-sync-vs-async-job-handling)
  * [When to Use Each Pattern](#when-to-use-each-pattern)
  * [The Async Job Lifecycle](#the-async-job-lifecycle)
* [2 · Redis as a Message Broker](#2-redis-as-a-message-broker)
  * [Redis Data Structures for Queuing](#redis-data-structures-for-queuing)
  * [Architecture](#architecture)
* [3 · Celery: Production Task Queues](#3-celery-production-task-queues)
  * [Celery Architecture](#celery-architecture)
  * [Key Concepts](#key-concepts)
* [4 · FastAPI + Celery Integration](#4-fastapi-celery-integration)
* [5 · Job Status Patterns: Polling vs Webhooks vs WebSockets](#5-job-status-patterns-polling-vs-webhooks-vs-websockets)
* [6 · Advanced Patterns](#6-advanced-patterns)
  * [Dead Letter Queues (DLQ)](#dead-letter-queues-dlq)
  * [Rate Limiting](#rate-limiting)
  * [Task Chaining & Workflows](#task-chaining-workflows)
* [7 · Choosing the Right Queue Technology](#7-choosing-the-right-queue-technology)
  * [Redis vs RabbitMQ as Broker](#redis-vs-rabbitmq-as-broker)
* [Knowledge Check](#knowledge-check)
  * [Question 1: When to Use a Task Queue](#question-1-when-to-use-a-task-queue)
  * [Question 2: 202 vs 200](#question-2-202-vs-200)
  * [Question 3: Worker Prefetch](#question-3-worker-prefetch)
  * [Question 4: acks_late](#question-4-acks_late)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Design a Job API](#exercise-1-design-a-job-api)
  * [Exercise 2: Priority Queue](#exercise-2-priority-queue)
  * [Exercise 3: Retry Strategy](#exercise-3-retry-strategy)
* [Summary](#summary)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors put everything in the API request handler: receive request → run model → return result. This works for 200ms API calls but catastrophically fails for 5-minute jobs. Seniors decouple the *request acceptance* from the *work execution* using message brokers, allowing the API to remain responsive while heavy work happens asynchronously on dedicated GPU workers.

**Mental Model:** A task queue is like a restaurant kitchen. The waiter (API server) takes your order and immediately moves to the next table. The order ticket goes on the rail (message broker). The chef (worker) picks up the ticket when ready. The waiter doesn't stand in the kitchen waiting — that would block service for everyone.

**Common Junior Pitfall:** Running a 10-minute inference job directly inside a FastAPI route handler. The HTTP connection times out after 30-60 seconds, the client gets a 504 Gateway Timeout, but the GPU keeps running the job with nowhere to send the result. You've wasted GPU time AND the user got an error.

---

In [ ]:
# Cell 0 — Install Dependencies
!pip install -q celery redis fakeredis fastapi uvicorn httpx

---
## 1 · The Two Patterns: Sync vs Async Job Handling

### When to Use Each Pattern

| Pattern | Latency | Example | Implementation |
|---------|---------|---------|----------------|
| **Synchronous** (NB27) | < 30s | LLM chat, classification | Return response directly |
| **SSE Streaming** (NB27) | 1-30s | LLM token streaming | Stream partial results |
| **Task Queue** (this notebook) | 30s - hours | Video gen, batch inference, fine-tuning | Enqueue + poll for result |

### The Async Job Lifecycle

```
1. CLIENT → POST /jobs        {"type": "generate_video", "prompt": "..."}
2. API    → 202 Accepted      {"job_id": "abc123", "status": "queued"}
3. API    → enqueue(job)      Pushes job to Redis/RabbitMQ
4. WORKER → dequeue(job)      Worker picks job from queue
5. WORKER → process(job)      GPU runs inference (5 minutes)
6. WORKER → store(result)     Saves to S3/DB, updates status
7. CLIENT → GET /jobs/abc123  Polls for completion
8. API    → 200 OK            {"status": "completed", "result_url": "..."}
```

**Key Insight:** The HTTP status code `202 Accepted` (not 200 OK) tells the client: "I received your request and will process it later." This is the standard REST convention for async operations.

In [ ]:
# Cell 1 — Simulate the async job pattern (no external services needed)

import uuid
import time
import threading
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional, Any
from datetime import datetime

class JobStatus(Enum):
    QUEUED = "queued"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"

@dataclass
class Job:
    job_id: str
    task_type: str
    payload: dict
    status: JobStatus = JobStatus.QUEUED
    result: Optional[Any] = None
    error: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    started_at: Optional[str] = None
    completed_at: Optional[str] = None
    progress: float = 0.0

class SimpleJobQueue:
    """In-memory job queue to demonstrate the pattern.
    In production, replace with Celery + Redis."""
    
    def __init__(self):
        self.jobs: dict[str, Job] = {}
        self.queue: list[str] = []
    
    def submit(self, task_type: str, payload: dict) -> Job:
        """Submit a job — returns immediately with job_id."""
        job = Job(
            job_id=str(uuid.uuid4())[:8],
            task_type=task_type,
            payload=payload,
        )
        self.jobs[job.job_id] = job
        self.queue.append(job.job_id)
        return job
    
    def get_status(self, job_id: str) -> dict:
        """Check job status — the polling endpoint."""
        job = self.jobs.get(job_id)
        if not job:
            return {"error": "Job not found"}
        return {
            "job_id": job.job_id,
            "status": job.status.value,
            "progress": job.progress,
            "result": job.result,
            "error": job.error,
        }
    
    def process_next(self, worker_fn):
        """Worker picks up next job (runs in background thread)."""
        if not self.queue:
            return None
        job_id = self.queue.pop(0)
        job = self.jobs[job_id]
        job.status = JobStatus.RUNNING
        job.started_at = datetime.now().isoformat()
        try:
            result = worker_fn(job)
            job.status = JobStatus.COMPLETED
            job.result = result
        except Exception as e:
            job.status = JobStatus.FAILED
            job.error = str(e)
        job.completed_at = datetime.now().isoformat()
        return job

# === Demo: Submit jobs and poll for results ===
queue = SimpleJobQueue()

# Step 1: API receives requests and returns immediately
print('=== Step 1: Submit Jobs (returns instantly) ===')
job1 = queue.submit("generate_video", {"prompt": "A cat riding a skateboard"})
job2 = queue.submit("batch_inference", {"dataset": "s3://data/images.tar", "count": 10000})
print(f'  Job 1: {job1.job_id} → status: {job1.status.value}')
print(f'  Job 2: {job2.job_id} → status: {job2.status.value}')
print(f'  (Both submitted in <1ms — client is NOT blocked)\n')

# Step 2: Worker processes jobs in background
print('=== Step 2: Worker Processes Jobs ===')
def video_worker(job):
    """Simulate 5-minute video generation (compressed to 1s for demo)."""
    for pct in [20, 40, 60, 80, 100]:
        job.progress = pct
        time.sleep(0.2)  # Simulate GPU work
    return {"video_url": "s3://outputs/cat-skateboard.mp4", "duration_s": 15}

processed = queue.process_next(video_worker)
print(f'  Processed: {processed.job_id} → {processed.status.value}')

# Step 3: Client polls for status
print(f'\n=== Step 3: Client Polls for Status ===')
status = queue.get_status(job1.job_id)
print(f'  GET /jobs/{job1.job_id} → {status}')

---
## 2 · Redis as a Message Broker

Redis is the most common broker for AI task queues because it's fast, simple, and you probably already have it for caching.

### Redis Data Structures for Queuing

| Structure | Use Case | Operation |
|-----------|----------|----------|
| **List** (LPUSH/BRPOP) | Simple FIFO queue | Push left, pop right (blocking) |
| **Sorted Set** (ZADD) | Priority queue | Score = priority, pop lowest |
| **Stream** (XADD/XREAD) | Event log with consumer groups | Multiple workers, delivery guarantees |
| **Pub/Sub** | Broadcast notifications | Real-time status updates |

### Architecture

```
┌──────────┐     ┌──────────┐     ┌────────────────────┐
│ FastAPI   │────→│  Redis    │←────│ Celery Worker 1    │ (CPU tasks)
│ (API)     │     │ (Broker)  │←────│ Celery Worker 2    │ (GPU tasks)
└──────────┘     │           │←────│ Celery Worker 3    │ (GPU tasks)
      ↑          │           │     └────────────────────┘
      │          │ (Results) │
   Client        └──────────┘
 polls for
  status
```

In [ ]:
# Cell 2 — Redis queue operations (using fakeredis for notebook portability)

import fakeredis
import json

r = fakeredis.FakeRedis(decode_responses=True)

# === Pattern 1: Simple FIFO Queue ===
print('=== Pattern 1: FIFO Queue (LPUSH/RPOP) ===')

# Producer: API server pushes jobs
jobs = [
    {"id": "job_001", "type": "inference", "model": "llama-3", "prompt": "Explain RAG"},
    {"id": "job_002", "type": "inference", "model": "llama-3", "prompt": "What is RLHF?"},
    {"id": "job_003", "type": "video_gen", "model": "sora-v2", "prompt": "Cat on skateboard"},
]

for job in jobs:
    r.lpush("ai:jobs:pending", json.dumps(job))
    print(f'  Enqueued: {job["id"]} ({job["type"]})')

print(f'  Queue length: {r.llen("ai:jobs:pending")}')

# Consumer: Worker pops jobs (FIFO: first in, first out)
print('\n  Worker processing:')
while r.llen("ai:jobs:pending") > 0:
    raw = r.rpop("ai:jobs:pending")  # In production: BRPOP (blocking)
    job = json.loads(raw)
    print(f'    Processing: {job["id"]} → {job["type"]}: {job["prompt"][:30]}')

# === Pattern 2: Priority Queue ===
print('\n=== Pattern 2: Priority Queue (Sorted Set) ===')

# Enterprise customers get priority 1, free tier gets priority 10
r.zadd("ai:jobs:priority", {json.dumps({"id": "free_001", "user": "free_user"}): 10})
r.zadd("ai:jobs:priority", {json.dumps({"id": "ent_001", "user": "enterprise_user"}): 1})
r.zadd("ai:jobs:priority", {json.dumps({"id": "pro_001", "user": "pro_user"}): 5})

print('  Processing order (lowest score = highest priority):')
while r.zcard("ai:jobs:priority") > 0:
    # ZPOPMIN: pop the lowest-score member (highest priority)
    result = r.zpopmin("ai:jobs:priority")
    member, score = result[0]
    job = json.loads(member)
    print(f'    Priority {int(score)}: {job["id"]} ({job["user"]})')

# === Pattern 3: Job Status Storage ===
print('\n=== Pattern 3: Job Status (Hash) ===')

# Store job status as a Redis hash (O(1) lookup by job_id)
r.hset("ai:job:job_001", mapping={
    "status": "running",
    "progress": "65",
    "started_at": "2024-03-15T10:30:00",
    "worker": "gpu-worker-02",
})

status = r.hgetall("ai:job:job_001")
print(f'  GET /jobs/job_001 → {status}')

# Set auto-expiry (clean up completed jobs after 24 hours)
r.expire("ai:job:job_001", 86400)
print(f'  TTL: {r.ttl("ai:job:job_001")}s (auto-cleanup after 24h)')

---
## 3 · Celery: Production Task Queues

Celery is the most widely used Python task queue. It handles:
- Task serialization & routing
- Worker process management
- Retries with exponential backoff
- Result storage & retrieval
- Scheduling (like cron, but integrated)
- Priority queues & rate limiting

### Celery Architecture

```
┌──────────────────────────────────────────┐
│                Celery                     │
│                                           │
│  [Client/API]  → sends task messages      │
│       ↓                                   │
│  [Broker]      → Redis or RabbitMQ        │
│       ↓          stores pending tasks     │
│  [Worker(s)]   → execute tasks            │
│       ↓          can run on separate VMs  │
│  [Result Backend] → Redis or DB           │
│                    stores return values   │
└──────────────────────────────────────────┘
```

### Key Concepts

| Concept | Definition | AI Example |
|---------|-----------|------------|
| **Task** | A Python function decorated with `@app.task` | `generate_video(prompt)` |
| **Broker** | Message queue (Redis/RabbitMQ) | Where tasks wait |
| **Worker** | Process that executes tasks | GPU server running inference |
| **Result Backend** | Where return values are stored | Redis hash with output URL |
| **Queue** | Named channel for routing tasks | `gpu_queue`, `cpu_queue` |
| **ETA/Countdown** | Delayed execution | "Run this in 5 minutes" |

In [ ]:
# Cell 3 — Complete Celery application for AI workloads
# Save as: tasks.py

celery_app_code = '''
from celery import Celery
from celery.utils.log import get_task_logger
import time
import os

# === Celery Configuration ===
app = Celery(
    "ai_tasks",
    broker=os.environ.get("CELERY_BROKER_URL", "redis://localhost:6379/0"),
    backend=os.environ.get("CELERY_RESULT_BACKEND", "redis://localhost:6379/1"),
)

# Configure task behavior
app.conf.update(
    # Serialization
    task_serializer="json",
    result_serializer="json",
    accept_content=["json"],
    
    # Timeouts
    task_soft_time_limit=600,    # Warn after 10 minutes
    task_time_limit=900,         # Kill after 15 minutes
    
    # Result management
    result_expires=86400,        # Clean up results after 24 hours
    
    # Worker settings
    worker_prefetch_multiplier=1, # Don't prefetch (for long GPU tasks)
    worker_concurrency=2,         # 2 concurrent tasks (= 2 GPU slots)
    
    # Task routing: different queues for different hardware
    task_routes={
        "ai_tasks.generate_video": {"queue": "gpu_heavy"},
        "ai_tasks.batch_inference": {"queue": "gpu_light"},
        "ai_tasks.process_document": {"queue": "cpu"},
    },
)

logger = get_task_logger(__name__)

# === Task Definitions ===

@app.task(
    bind=True,                    # Access self for status updates
    max_retries=3,                # Retry up to 3 times
    default_retry_delay=60,       # Wait 60s before retry
    acks_late=True,               # Only ACK after task completes
    reject_on_worker_lost=True,   # Re-queue if worker crashes
)
def generate_video(self, prompt: str, duration_s: int = 10):
    """Generate a video from a text prompt.
    This runs on a GPU worker and may take 5-15 minutes.
    """
    logger.info(f"Starting video generation: {prompt[:50]}")
    
    try:
        total_frames = duration_s * 24  # 24 fps
        for frame in range(total_frames):
            # Update progress (visible via API polling)
            self.update_state(
                state="PROGRESS",
                meta={
                    "current": frame + 1,
                    "total": total_frames,
                    "percent": round((frame + 1) / total_frames * 100, 1),
                }
            )
            time.sleep(0.01)  # Simulate GPU rendering
        
        # Upload result to S3
        result_url = f"s3://ai-outputs/videos/{self.request.id}.mp4"
        return {
            "status": "completed",
            "video_url": result_url,
            "duration_s": duration_s,
            "frames": total_frames,
        }
    except Exception as exc:
        # Retry with exponential backoff
        raise self.retry(exc=exc, countdown=60 * (2 ** self.request.retries))


@app.task(bind=True, max_retries=2)
def batch_inference(self, dataset_path: str, model_name: str = "resnet50"):
    """Run batch inference on a dataset of images."""
    logger.info(f"Batch inference: {dataset_path} with {model_name}")
    
    # Simulate processing 10K images
    total_images = 10000
    batch_size = 256
    results = []
    
    for batch_start in range(0, total_images, batch_size):
        batch_end = min(batch_start + batch_size, total_images)
        self.update_state(
            state="PROGRESS",
            meta={"processed": batch_end, "total": total_images}
        )
        time.sleep(0.01)  # Simulate batch processing
    
    return {
        "total_processed": total_images,
        "results_path": f"s3://ai-outputs/batch/{self.request.id}.parquet",
    }


@app.task(bind=True)
def process_document(self, document_url: str):
    """OCR + RAG indexing pipeline (CPU-intensive)."""
    steps = ["download", "ocr", "chunk", "embed", "index"]
    for i, step in enumerate(steps):
        self.update_state(
            state="PROGRESS",
            meta={"step": step, "step_num": i + 1, "total_steps": len(steps)}
        )
        time.sleep(0.1)  # Simulate processing
    
    return {
        "chunks": 47,
        "collection": "documents",
        "indexed": True,
    }
'''

print('=== Celery Task Definitions (tasks.py) ===')
print(celery_app_code)
print('\n--- Starting Workers ---')
print('# Start GPU worker (listens to gpu_heavy and gpu_light queues):')
print('celery -A tasks worker -Q gpu_heavy,gpu_light -c 2 --hostname=gpu@%h')
print()
print('# Start CPU worker (listens to cpu queue):')
print('celery -A tasks worker -Q cpu -c 8 --hostname=cpu@%h')
print()
print('# Monitor all workers:')
print('celery -A tasks flower  # Web UI at http://localhost:5555')

---
## 4 · FastAPI + Celery Integration

This is the production pattern for AI backends: FastAPI handles HTTP, Celery handles heavy work.

In [ ]:
# Cell 4 — Complete FastAPI + Celery API Server
# Save as: api_server.py

api_server_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from celery.result import AsyncResult
from tasks import app as celery_app, generate_video, batch_inference, process_document
from typing import Optional
import uuid

api = FastAPI(title="AI Job Queue API", version="1.0")

# === Request/Response Models ===

class VideoRequest(BaseModel):
    prompt: str = Field(..., min_length=5, max_length=1000)
    duration_s: int = Field(default=10, ge=1, le=60)

class BatchInferenceRequest(BaseModel):
    dataset_path: str
    model_name: str = "resnet50"

class JobResponse(BaseModel):
    job_id: str
    status: str
    message: str

class JobStatusResponse(BaseModel):
    job_id: str
    status: str
    progress: Optional[dict] = None
    result: Optional[dict] = None
    error: Optional[str] = None

# === Endpoints ===

@api.post("/jobs/video", response_model=JobResponse, status_code=202)
async def submit_video_job(req: VideoRequest):
    """Submit a video generation job. Returns immediately."""
    task = generate_video.delay(prompt=req.prompt, duration_s=req.duration_s)
    return JobResponse(
        job_id=task.id,
        status="queued",
        message=f"Video generation queued. Poll GET /jobs/{task.id} for status.",
    )

@api.post("/jobs/batch-inference", response_model=JobResponse, status_code=202)
async def submit_batch_job(req: BatchInferenceRequest):
    """Submit a batch inference job."""
    task = batch_inference.delay(dataset_path=req.dataset_path, model_name=req.model_name)
    return JobResponse(
        job_id=task.id,
        status="queued",
        message=f"Batch inference queued. Poll GET /jobs/{task.id} for status.",
    )

@api.get("/jobs/{job_id}", response_model=JobStatusResponse)
async def get_job_status(job_id: str):
    """Poll for job status and progress."""
    result = AsyncResult(job_id, app=celery_app)
    
    response = JobStatusResponse(job_id=job_id, status=result.status)
    
    if result.status == "PROGRESS":
        response.progress = result.info  # Contains percent, step, etc.
    elif result.status == "SUCCESS":
        response.result = result.result
    elif result.status == "FAILURE":
        response.error = str(result.result)
    
    return response

@api.delete("/jobs/{job_id}")
async def cancel_job(job_id: str):
    """Cancel a running or queued job."""
    celery_app.control.revoke(job_id, terminate=True)
    return {"job_id": job_id, "status": "cancelled"}

@api.get("/health")
async def health_check():
    """Check API + worker health."""
    inspector = celery_app.control.inspect()
    workers = inspector.active_queues() or {}
    return {
        "api": "healthy",
        "workers": len(workers),
        "queues": list(set(
            q["name"] for w in workers.values() for q in w
        )) if workers else [],
    }
'''

print('=== FastAPI + Celery API Server (api_server.py) ===')
print(api_server_code)
print('\n--- Usage ---')
print('# 1. Start Redis:  docker run -d -p 6379:6379 redis:7')
print('# 2. Start worker: celery -A tasks worker -Q gpu_heavy -c 2')
print('# 3. Start API:    uvicorn api_server:api --port 8000')
print()
print('# Submit a job:')
print('curl -X POST http://localhost:8000/jobs/video \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"prompt": "A cat on a skateboard", "duration_s": 10}\'')
print()
print('# Poll for status:')
print('curl http://localhost:8000/jobs/<job_id>')

---
## 5 · Job Status Patterns: Polling vs Webhooks vs WebSockets

| Pattern | How It Works | Pros | Cons | Best For |
|---------|-------------|------|------|----------|
| **Polling** | Client repeatedly calls `GET /jobs/{id}` | Simple, stateless | Wastes requests, delayed updates | Most AI backends |
| **Webhooks** | Server POSTs to client URL on completion | Instant notification, efficient | Client needs public endpoint | B2B integrations |
| **WebSocket** | Persistent bidirectional connection | Real-time updates | Complex, stateful | Live dashboards |
| **SSE** (NB27) | Server pushes updates over HTTP | Simpler than WebSocket | One-directional | Progress bars |

In [ ]:
# Cell 5 — Webhook callback pattern

webhook_code = '''
# In your Celery task — call webhook on completion:

import httpx

@app.task(bind=True)
def generate_video_with_webhook(
    self, 
    prompt: str, 
    webhook_url: str = None,   # Client provides their callback URL
):
    """Generate video and notify client via webhook when done."""
    # ... do the work ...
    result = {"video_url": "s3://...", "duration": 10}
    
    # Notify client via webhook
    if webhook_url:
        try:
            httpx.post(webhook_url, json={
                "job_id": self.request.id,
                "status": "completed",
                "result": result,
            }, timeout=10)
        except httpx.HTTPError:
            pass  # Don't fail the job if webhook fails
    
    return result

# Client submits with webhook URL:
# POST /jobs/video
# {
#   "prompt": "Cat on skateboard",
#   "webhook_url": "https://client-app.com/webhooks/ai-jobs"
# }
'''

print('=== Webhook Callback Pattern ===')
print(webhook_code)

# Demonstrate polling with exponential backoff
print('\n=== Smart Polling with Exponential Backoff ===')

import time

def poll_with_backoff(get_status_fn, job_id, max_wait=120):
    """Production polling: start fast, slow down over time."""
    interval = 1  # Start at 1 second
    total_waited = 0
    
    while total_waited < max_wait:
        status = get_status_fn(job_id)
        print(f'  [{total_waited:3d}s] Poll → status: {status["status"]}, '
              f'progress: {status.get("progress", "-")}%')
        
        if status["status"] in ("completed", "failed"):
            return status
        
        time.sleep(0.01)  # Simulated (real: time.sleep(interval))
        total_waited += interval
        interval = min(interval * 1.5, 30)  # Cap at 30 seconds
    
    return {"status": "timeout"}

# Simulate a job that completes after 30 seconds
def mock_status(job_id):
    mock_status.call_count += 1
    if mock_status.call_count < 8:
        return {"status": "running", "progress": mock_status.call_count * 13}
    return {"status": "completed", "result": {"url": "s3://output.mp4"}}
mock_status.call_count = 0

print('\nPolling with exponentially increasing intervals:')
result = poll_with_backoff(mock_status, "job_123")
print(f'\nFinal result: {result}')

---
## 6 · Advanced Patterns

### Dead Letter Queues (DLQ)
When a task fails after all retries, it goes to a DLQ for manual investigation.

### Rate Limiting
Prevent workers from overwhelming external APIs (e.g., OpenAI's rate limits).

### Task Chaining & Workflows
Chain multiple tasks into a pipeline that executes sequentially.

In [ ]:
# Cell 6 — Advanced Celery patterns

print('=== 1. Task Chaining (Celery Workflows) ===')
chain_code = '''
from celery import chain, group, chord

# chain(): Run tasks sequentially (output of A → input of B)
rag_pipeline = chain(
    download_document.s("https://example.com/paper.pdf"),
    extract_text.s(),           # receives download result
    chunk_document.s(),         # receives extracted text
    generate_embeddings.s(),    # receives chunks
    store_in_vectordb.s(),      # receives embeddings
)
result = rag_pipeline.delay()  # Execute the whole chain

# group(): Run tasks in PARALLEL
batch = group(
    classify_image.s(img) for img in image_urls
)
results = batch.delay()  # All run simultaneously

# chord(): Parallel tasks + callback when ALL complete
pipeline = chord(
    group(classify_image.s(img) for img in image_urls),
    aggregate_results.s()  # Runs after ALL images classified
)
'''
print(chain_code)

print('\n=== 2. Rate Limiting ===')
rate_limit_code = '''
@app.task(rate_limit="10/m")  # Max 10 calls per minute
def call_openai_api(prompt: str):
    """Rate-limited to stay within OpenAI's rate limits."""
    response = openai.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Workers will automatically throttle to 10 calls/minute
'''
print(rate_limit_code)

print('\n=== 3. Dead Letter Queue (DLQ) ===')
dlq_code = '''
from celery.signals import task_failure

@task_failure.connect
def handle_task_failure(sender=None, task_id=None, exception=None, **kwargs):
    """Move permanently failed tasks to a dead letter queue."""
    if sender.request.retries >= sender.max_retries:
        # Push to DLQ for manual investigation
        redis_client.lpush("ai:dead_letter_queue", json.dumps({
            "task_id": task_id,
            "task_name": sender.name,
            "error": str(exception),
            "args": sender.request.args,
            "failed_at": datetime.now().isoformat(),
        }))
        # Alert on-call engineer
        slack.post(f"🚨 Task {task_id} permanently failed: {exception}")
'''
print(dlq_code)

---
## 7 · Choosing the Right Queue Technology

| Technology | Language | Best For | Broker |
|-----------|---------|---------|--------|
| **Celery** | Python | Most Python AI backends | Redis or RabbitMQ |
| **Dramatiq** | Python | Simpler alternative to Celery | Redis or RabbitMQ |
| **BullMQ** | Node.js | JS/TS backends | Redis |
| **RabbitMQ** (direct) | Any | Complex routing, multi-language | Self (AMQP) |
| **AWS SQS** | Any | AWS-native, zero ops | Managed |
| **GCP Cloud Tasks** | Any | GCP-native | Managed |

### Redis vs RabbitMQ as Broker

| | Redis | RabbitMQ |
|---|---|---|
| Speed | Faster | Fast |
| Durability | Optional (AOF/RDB) | Always (disk-backed) |
| Routing | Simple (queue names) | Complex (exchanges, bindings) |
| Monitoring | Basic | Excellent (Management UI) |
| **Use when** | You already have Redis for caching | You need guaranteed delivery |

**Senior Guidance:** Start with **Celery + Redis**. It covers 95% of AI backend needs. Switch to RabbitMQ only if you need complex routing or absolute message durability guarantees.

---
## Knowledge Check

### Question 1: When to Use a Task Queue
You're building an API that classifies images. Each inference takes 200ms. Do you need a task queue?

<details>
<summary>Click to reveal answer</summary>

**No.** 200ms is well within HTTP timeout limits. Return the result synchronously in the API response. Task queues add unnecessary complexity for fast operations. Use queues only when tasks take >30 seconds or when you need to decouple submission from execution (e.g., rate limiting, priority ordering).
</details>

### Question 2: 202 vs 200
Why does the video generation endpoint return HTTP 202 instead of 200?

<details>
<summary>Click to reveal answer</summary>

**202 Accepted** means "I received your request and will process it later" — the work hasn't started yet. **200 OK** means "here is the result." Using 202 is the REST convention for asynchronous operations. The client knows to poll `GET /jobs/{id}` for the actual result.
</details>

### Question 3: Worker Prefetch
Why set `worker_prefetch_multiplier=1` for GPU task workers?

<details>
<summary>Click to reveal answer</summary>

Prefetch means a worker reserves multiple tasks in advance. For fast CPU tasks, this is efficient. For slow GPU tasks (5-15 minutes each), prefetching means one worker reserves tasks that another idle worker could process. Setting it to 1 ensures fair distribution: each worker only takes the next task when it finishes the current one.
</details>

### Question 4: acks_late
What happens if a worker crashes while processing a task with `acks_late=True` vs without it?

<details>
<summary>Click to reveal answer</summary>

**Without `acks_late`** (default): The task is acknowledged (removed from queue) immediately when the worker starts. If the worker crashes, the task is **lost forever**. **With `acks_late=True`**: The task is acknowledged only after it completes successfully. If the worker crashes, the task remains in the queue and gets picked up by another worker. This is essential for expensive GPU tasks.
</details>

---
## Practical Practice

### Exercise 1: Design a Job API
You're building an AI document processing service. Users upload PDFs, and the system:
1. OCRs the document (2 minutes)
2. Chunks the text (10 seconds)
3. Generates embeddings (30 seconds)
4. Indexes into a vector DB (5 seconds)

Design the REST API endpoints (submit, status, cancel) and the Celery task chain.

### Exercise 2: Priority Queue
Implement a priority queue where enterprise customers' jobs run before free-tier customers. Use Redis sorted sets. Write the enqueue and dequeue functions.

### Exercise 3: Retry Strategy
Your video generation task fails 20% of the time due to GPU OOM errors. Design a retry strategy with exponential backoff that:
- Retries up to 3 times
- Waits 1 min, 4 min, 16 min between retries
- Moves to DLQ after all retries exhausted
- Alerts on-call engineer via Slack

In [ ]:
# ===== EXERCISE SOLUTIONS (try yourself first!) =====

print('=== Exercise 1: Document Processing API ===')
print('''
# Endpoints:
POST /documents          → 202 {"job_id": "...", "status": "queued"}
GET  /documents/{job_id} → 200 {"status": "processing", "step": "ocr", "progress": 45}
DELETE /documents/{job_id} → 200 {"status": "cancelled"}

# Celery chain:
from celery import chain

pipeline = chain(
    ocr_document.s(document_url),        # 2 min
    chunk_text.s(),                       # 10 sec
    generate_embeddings.s(),              # 30 sec
    index_in_vectordb.s(collection_id),   # 5 sec
)
result = pipeline.delay()
''')

print('\n=== Exercise 2: Priority Queue ===')
import fakeredis, json
r = fakeredis.FakeRedis(decode_responses=True)

def enqueue_with_priority(job: dict, priority: int):
    """Lower number = higher priority. Enterprise=1, Pro=5, Free=10."""
    r.zadd("ai:priority_queue", {json.dumps(job): priority})

def dequeue_highest_priority():
    """Pop the highest-priority (lowest score) job."""
    result = r.zpopmin("ai:priority_queue")
    if result:
        return json.loads(result[0][0]), int(result[0][1])
    return None, None

enqueue_with_priority({"id": "free_1", "task": "inference"}, priority=10)
enqueue_with_priority({"id": "ent_1", "task": "inference"}, priority=1)
enqueue_with_priority({"id": "pro_1", "task": "inference"}, priority=5)

print('Dequeue order:')
for _ in range(3):
    job, prio = dequeue_highest_priority()
    print(f'  Priority {prio}: {job["id"]}')

print('\n=== Exercise 3: Retry Strategy ===')
print('''
@app.task(
    bind=True,
    max_retries=3,
    autoretry_for=(torch.cuda.OutOfMemoryError, RuntimeError),
    retry_backoff=60,       # Base: 60 seconds
    retry_backoff_max=960,  # Cap: 16 minutes
    retry_jitter=True,      # Add randomness to prevent thundering herd
)
def generate_video(self, prompt, duration):
    try:
        return run_generation(prompt, duration)
    except torch.cuda.OutOfMemoryError as exc:
        if self.request.retries >= self.max_retries:
            # Move to DLQ and alert
            push_to_dlq(self.request.id, str(exc))
            slack_alert(f"Video gen {self.request.id} failed permanently")
        raise  # Celery handles the retry via autoretry_for
''')

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| Async job pattern | Submit → Queue → Worker → Poll for result |
| Redis as broker | Lists (FIFO), sorted sets (priority), hashes (status) |
| Celery | Production task queue with retries, routing, rate limiting |
| FastAPI + Celery | API returns 202, worker processes async, client polls |
| Status patterns | Polling (simple), webhooks (efficient), WebSocket (real-time) |
| Advanced | Task chaining, priority queues, DLQs, rate limiting |

**Connections:** SSE streaming for fast tasks → NB27 | Model deployment containers → NB33 | CI/CD for workers → NB31 | Monitoring workers → NB35  
**Next →** `30_inference_optimization_gpu.ipynb` — LLM Inference Optimization & GPU Economics